# Category analysis: ten category-boundary examples
Standalone VS Code OpenRouter experiment for `llama-4-scout`, `gpt-oss-120b`, and `muse-glimmer`. Runs only the new corrected prompt; leaves the original notebooks and results intact.

Select the project's `.venv` Python kernel. If dependencies are missing, run `%pip install pandas scikit-learn openai python-dotenv tqdm` in a new cell. Supply `OPENROUTER_API_KEY` in your environment or local `.env.local` / `.env` file. No Colab, cloning, or automatic package installation is required.

The revised ten examples are verified records from `prompt_development.csv`, selected to address the latest run's category-boundary errors. The old eight error-review records remain excluded, so this follow-up still evaluates exactly the same 172 permits. New examples do not overlap evaluation descriptions. Because evaluation errors informed both prompt revisions, these scores are diagnostic; a fresh unseen split is needed to confirm generalization.

OpenRouter results are stored in `category_results_fewshot8_openrouter`, separate from the failed Jetstream run. Default concurrency: 8 requests; adjust `MAX_WORKERS` if rate limits occur. Descriptions are preserved in full.


In [1]:
import hashlib
import json
import os
from pathlib import Path
import importlib
import openrouter_experiment
importlib.reload(openrouter_experiment)
from openrouter_experiment import MODEL_IDS, OpenRouterPromptClient, run_checkpointed

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from tqdm.auto import tqdm
from road_ai import (
    DEFAULT_CATEGORIES, classification_prompt, jetstream_client,
    load_fixed_splits, run_experiment, score_results,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "category_splits").is_dir():
    raise RuntimeError("Open this notebook from the pgh_road_ai_starter project directory.")
load_dotenv(PROJECT_ROOT / ".env.local", override=True)
load_dotenv(PROJECT_ROOT / ".env", override=False)
CATEGORIES = DEFAULT_CATEGORIES
LABEL_COL = "work_type"
MODELS = ["llama-4-scout", "gpt-oss-120b", "muse-glimmer"]
OUTPUT_DIR = PROJECT_ROOT / "category_results_fewshot8_openrouter"
MAX_WORKERS = 8
REQUEST_TIMEOUT = 45
prompt_dev, original_test = load_fixed_splits(LABEL_COL, PROJECT_ROOT / "category_splits")


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Revised examples: activity, staging, access, and disposal
Ten development-set examples target reviewed confusions, including contrasting siding-material storage versus shingle-removal disposal and utility inspection versus HVAC placement. “Wrong alternatives” describe errors observed on related evaluation records, not claimed model predictions on these development examples. Recorded labels can depend on omitted permit context; exceptions are explicitly acknowledged.


In [2]:
REVIEW_CORRECTIONS = [('DOMI-CS-2024-00203', 'MATERIALS / BARRICADE', 'MACHINERY', 'Reviewed errors treated sewer inspection, cable pulling and utility work as material handling or generic closures. Here CCTV inspection is an operational utility task, not delivery or storage of pipes. Choose MACHINERY. Sewer CCTV, lining/CIPP, manhole inspection/duct proofing and cable pulling share this operational pattern even when no machine is named. Do not infer BARRICADE merely because work occurs in the street.'), ('DOMI-CS-2026-03701', 'MACHINERY / MATERIALS', 'CRANE', 'Reviewed errors classified HVAC installation as ordinary machinery or material delivery. This development permit is CRANE although the text does not explicitly say crane or lift. Large building HVAC placement is a dataset-specific lifting precedent. Contrast operational utility inspection (MACHINERY) and ground-level storage/delivery (MATERIALS). Infer the category cautiously; do not claim the description explicitly names a crane.'), ('DOMI-CS-2023-10689', 'MACHINERY / BARRICADE', 'MATERIALS', 'Reviewed errors over-weighted machines or street occupancy when material staging was also stated. This labeled example says staging materials AND equipment and is MATERIALS: storage/laydown is the permit function, rather than equipment actively performing an inspection or lift. Mentioning equipment does not automatically override material staging; a street location alone does not establish BARRICADE.'), ('DOMI-CS-2021-07474', 'DEMOLITION DUMPSTER / MACHINERY', 'MATERIALS', 'Reviewed siding and renovation descriptions were ambiguous between disposal and installing new materials. Here the explicit purpose is space to STAGE MATERIALS for two weeks, so MATERIALS. Contrast the roof-shingle-replacement disposal precedent below. Replacement alone is not a universal dumpster rule; explicit storage/delivery purpose is stronger evidence than the project title.'), ('DOMI-CS-2024-08603', 'DEMOLITION DUMPSTER / MATERIALS', 'BARRICADE', 'Reviewed mixed construction sites were classified from one incidental noun such as crane, dumpster or materials. Here the main request is an enclosing barricade and restricted lane/site laydown area; the dumpster and stored materials sit WITHIN it. Choose BARRICADE for that overall site protection/occupation function. This does not mean every street dumpster or temporary unloading lane closure is BARRICADE.'), ('DOMI-CS-2021-13673', 'MATERIALS / BARRICADE', 'DEMOLITION DUMPSTER', 'Reviewed clean-outs were treated as material handling or generic closures. This sparse BUILDING CLEANOUT permit is labeled DEMOLITION DUMPSTER. Clean-out means removal/disposal of unwanted contents rather than storage of incoming construction supplies. Dumpster need is dataset precedent, not an explicit noun here. Interior strip-out and renovation debris differ from protected whole-site demolition closures; demolition wording alone cannot decide the category.'), ('DOMI-CS-2022-09453', 'MATERIALS / BARRICADE', 'DEMOLITION DUMPSTER', 'Reviewed shingle/siding replacement and interior removal were mistaken for incoming materials or barricades. This roof-shingle-replacement permit is DEMOLITION DUMPSTER: a replacement/removal disposal precedent, with no rooftop lift or material-storage purpose stated. Contrast siding with explicit material staging (MATERIALS) and roof/HVAC lifting (CRANE). Do not invent a stated container or apply this label to every renovation.'), ('DOMI-CS-2023-10972', 'MACHINERY / MATERIALS', 'SCAFFOLD', 'Reviewed gutter and facade repair cases were classified as materials or unspecified equipment. This short box-gutter repair permit is SCAFFOLD, a dataset-specific building-access precedent despite no explicit scaffold noun. Contrast explicit man-lift/JLG operations (usually MACHINERY) and HVAC placement (CRANE). The reviewed data contain exceptions, so facade/inspection wording alone cannot establish a scaffold; report uncertainty instead of claiming unstated equipment.'), ('DOMI-CS-2025-04182', 'SCAFFOLD / BARRICADE', 'MACHINERY', 'Reviewed facade inspections were confused with scaffolding or generic closures. Here a JLG operates from the curb lane for curtain-wall inspection and window cleaning: mobile access equipment is the operational staging purpose, so MACHINERY. Flaggers and street occupancy are supporting controls, not enough to change the category to BARRICADE. Contrast erected scaffold access and the sparse box-gutter SCAFFOLD precedent; facade work alone does not identify its access equipment.'), ('DOMI-CS-2023-11892', 'BARRICADE / CRANE', 'MATERIALS', 'Reviewed material staging was confused with barricades because a travel lane was occupied. This permit explicitly blocks a lane temporarily to unload/load supplies and move them into a building, so MATERIALS. The lane closure supports delivery rather than a protected construction-site enclosure. No rooftop or heavy-unit lifting is stated, so do not invent CRANE from truck unloading. Contrast the enclosing barricade example and HVAC-placement CRANE precedent.')]

LEGACY_EXCLUDED_IDS = ['DOMI-CS-2020-01589', 'DOMI-CS-2021-12554', 'DOMI-CS-2022-08484', 'DOMI-CS-2022-10074', 'DOMI-CS-2023-03525', 'DOMI-CS-2024-05362', 'DOMI-CS-2025-04623', 'DOMI-CS-2024-15300']

examples = []
for permit_id, wrong, correct, explanation in REVIEW_CORRECTIONS:
    matches = prompt_dev[prompt_dev.permit_id.eq(permit_id)]
    assert len(matches) == 1, permit_id
    row = matches.iloc[0]
    assert row[LABEL_COL] == correct, permit_id
    examples.append({
        "permit_id": permit_id, "description": row.work_description,
        "wrong_alternatives": wrong, "category": correct,
        "explanation": explanation, "source": "prompt_development.csv",
    })
assert len(examples) == 10

SYSTEM_PROMPT = classification_prompt(CATEGORIES) + "\nKeep reason under 20 words.\n" + (
    "Classify the permit's main function: operational equipment, lifting, material "
    "storage/delivery, disposal, building-access scaffold, or protected site closure. "
    "Use the whole description, not one equipment/material noun. A temporary street "
    "closure can accompany any category. Do not invent a rigid keyword hierarchy. "
    "Use the ten verified precedents below when wording is sparse; do not claim "
    "unstated equipment is explicit. Permit labels may depend on omitted context.\n"
) + "\n".join(
    f"{i}. Description: {e['description']}\n"
    f"Confusable alternatives: {e['wrong_alternatives']}; correct: {e['category']}.\n"
    f"Boundary lesson: {e['explanation']}"
    for i, e in enumerate(examples, start=1)
)
normalize = lambda s: " ".join(str(s).upper().split())
example_descriptions = {normalize(e["description"]) for e in examples}
legacy_descriptions = set(original_test[
    original_test.permit_id.isin(LEGACY_EXCLUDED_IDS)
].work_description.map(normalize))
test = original_test[
    ~original_test.work_description.map(normalize).isin(legacy_descriptions)
].copy()
assert len(test) == 172
assert not set(test.work_description.map(normalize)) & example_descriptions
assert not set(test.permit_id) & {e["permit_id"] for e in examples}
display(pd.DataFrame(examples))
print(f"Evaluation records: {len(test)}; development examples: {len(examples)}")
print(SYSTEM_PROMPT)


,permit_id,description,wrong_alternatives,category,explanation,source
0,DOMI-CS-2024-00203,CCTV INSPECTION OF PWSA SEWERS.,MATERIALS / BARRICADE,MACHINERY,"Reviewed errors treated sewer inspection, cabl...",prompt_development.csv
1,DOMI-CS-2026-03701,CLIMATECH INC. INSTALLING HVAC EQUIPMENT AT TH...,MACHINERY / MATERIALS,CRANE,Reviewed errors classified HVAC installation a...,prompt_development.csv
2,DOMI-CS-2023-10689,STAGING MATERIALS AND EQUIPMENT FOR EDGERTON A...,MACHINERY / BARRICADE,MATERIALS,Reviewed errors over-weighted machines or stre...,prompt_development.csv
3,DOMI-CS-2021-07474,REPLACING SIDING AT 5613 HOWE STREET. WILL NE...,DEMOLITION DUMPSTER / MACHINERY,MATERIALS,Reviewed siding and renovation descriptions we...,prompt_development.csv
4,DOMI-CS-2024-08603,THE PROJECT IS LOCATED AT THE HIGHMARK BUILDIN...,DEMOLITION DUMPSTER / MATERIALS,BARRICADE,Reviewed mixed construction sites were classif...,prompt_development.csv
5,DOMI-CS-2021-13673,BUILDING CLEANOUT,MATERIALS / BARRICADE,DEMOLITION DUMPSTER,Reviewed clean-outs were treated as material h...,prompt_development.csv
6,DOMI-CS-2022-09453,ROOF REPLACEMENT SHINGLES ONLY,MATERIALS / BARRICADE,DEMOLITION DUMPSTER,Reviewed shingle/siding replacement and interi...,prompt_development.csv
7,DOMI-CS-2023-10972,REPAIRING BOX GUTTERS,MACHINERY / MATERIALS,SCAFFOLD,Reviewed gutter and facade repair cases were c...,prompt_development.csv
8,DOMI-CS-2025-04182,BUILDING EXTERIOR CURTAIN WALL INSPECTIONS & W...,SCAFFOLD / BARRICADE,MACHINERY,Reviewed facade inspections were confused with...,prompt_development.csv
9,DOMI-CS-2023-11892,WE NEED TO BLOCK ONE LANE OF GRANT ST. TO UNLO...,BARRICADE / CRANE,MATERIALS,Reviewed material staging was confused with ba...,prompt_development.csv


Evaluation records: 172; development examples: 10
You classify Pittsburgh street-closure work descriptions into exactly one allowed construction-staging category.
Allowed categories: MACHINERY, DEMOLITION DUMPSTER, CRANE, BARRICADE, MATERIALS, SCAFFOLD.
Use only evidence in the description. Do not invent facts. If wording is ambiguous, choose the single best allowed category.
Return valid JSON only: {"category":"<allowed category>","confidence":0.0,"reason":"<brief evidence-based reason>"}.
Confidence must be between 0 and 1.

Keep reason under 20 words.
Classify the permit's main function: operational equipment, lifting, material storage/delivery, disposal, building-access scaffold, or protected site closure. Use the whole description, not one equipment/material noun. A temporary street closure can accompany any category. Do not invent a rigid keyword hierarchy. Use the ten verified precedents below when wording is sparse; do not claim unstated equipment is explicit. Permit labels may

## Concurrent OpenRouter run
Uses eight concurrent requests per model, short JSON responses, low reasoning effort where supported, a 45-second timeout per attempt, and one SDK retry for transient failures. The eight explanations stay in the prompt; duplicated example JSON is removed. No description text is truncated.

Each model tests one pending record before launching the rest. Successes are reused; failed records are retried on rerun. Five consecutive failures stop scheduling new requests. Completed requests are checkpointed; provider, model IDs, prompt, and data identify the run. Latency routing requests the fastest available provider, but speed depends on provider load and account limits. All three models are evaluated on the same 172 records.

Scout uses prompt-only JSON instructions because its available non-BYOK endpoints reject the `response_format` parameter. The other two models request JSON-object responses.


In [3]:
import time

run_settings = {
    "provider": "openrouter", "model_ids": MODEL_IDS,
    "prompt": SYSTEM_PROMPT, "reasoning_effort": "low",
    "max_tokens": {"llama-4-scout": 256, "gpt-oss-120b": 1024, "muse-glimmer": 1024},
    "routing": "latency", "temperature": 0, "response_format": {"llama-4-scout": None, "gpt-oss-120b": "json_object", "muse-glimmer": "json_object"},
}
fingerprint = hashlib.sha256(
    (json.dumps(run_settings, sort_keys=True) + test.to_csv(index=False)).encode()
).hexdigest()[:16]
RUN_DIR = OUTPUT_DIR / fingerprint
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "run_settings.json").write_text(json.dumps(run_settings, indent=2))
(RUN_DIR / "system_prompt.txt").write_text(SYSTEM_PROMPT)
(RUN_DIR / "examples.json").write_text(json.dumps(examples, indent=2))
test.to_csv(RUN_DIR / "evaluation_records.csv", index=False)
client = OpenRouterPromptClient(SYSTEM_PROMPT, timeout=REQUEST_TIMEOUT)
all_results = []
started = time.perf_counter()
for model in MODELS:
    checkpoint = RUN_DIR / f"{model}.jsonl"
    with tqdm(total=len(test), desc=model) as progress:
        run = run_checkpointed(
            test, LABEL_COL, model, CATEGORIES, client, checkpoint,
            max_workers=MAX_WORKERS, on_progress=progress.update,
        )
    run["model"] = model
    run["model_id"] = MODEL_IDS[model]
    run["provider"] = "openrouter"
    run["prompt_condition"] = "boundary_review_ten"
    run["n_examples"] = len(examples)
    all_results.append(run)
    # Save each completed model so interrupted runs retain usable results.
    pd.concat(all_results, ignore_index=True).to_csv(RUN_DIR / "classification_results.csv", index=False)
results = pd.concat(all_results, ignore_index=True)
print(f"Elapsed: {(time.perf_counter() - started) / 60:.1f} minutes")
print("Results saved to:", RUN_DIR)


muse-glimmer: 100%|██████████| 172/172 [01:49<00:00,  1.58it/s]

Elapsed: 2.6 minutes
Results saved to: /Users/maxhowarth/Desktop/pgh_road_ai_starter/category_results_fewshot8_openrouter/d1829a1157bb0c42


## Metrics and error review
Failures stay in the scoring denominator. Original notebook summary scores used 180 records and are not directly comparable with these 172-record scores. If the original raw results CSV is available locally, the next cell compares both prompts on exactly the same 172 records without rerunning the original experiments.


In [4]:
summary_rows, matrix_rows = [], []
for model, group in results.groupby("model", sort=False):
    scores = score_results(group, CATEGORIES)
    summary_rows.append({
        "model": model, "prompt_condition": "boundary_review_ten", "n_examples": len(examples),
        "n_scored": scores["n_scored"], "accuracy": scores["accuracy"],
        "macro_f1": scores["macro_f1"], "failed_predictions": scores["n_failures"],
        "prompt_tokens": group.prompt_tokens.sum(),
        "completion_tokens": group.completion_tokens.sum(),
    })
    display(model, scores["confusion_matrix"])
    for actual, row in scores["confusion_matrix"].iterrows():
        for predicted, count in row.items():
            matrix_rows.append({"model": model, "actual": actual, "predicted": predicted, "count": int(count)})
summary = pd.DataFrame(summary_rows)
summary.to_csv(RUN_DIR / "classification_summary.csv", index=False)
pd.DataFrame(matrix_rows).to_csv(RUN_DIR / "confusion_matrices.csv", index=False)
display(summary)
errors = results[results.actual.ne(results.predicted)].copy()
errors.to_csv(RUN_DIR / "error_review.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(errors[["model", "permit_id", "description", "actual", "predicted", "confidence", "reason", "error"]])


'llama-4-scout'

,MACHINERY,DEMOLITION DUMPSTER,CRANE,BARRICADE,MATERIALS,SCAFFOLD,__FAILED__
MACHINERY,22,0,1,1,0,0,4
DEMOLITION DUMPSTER,0,24,0,0,0,0,5
CRANE,2,1,17,1,1,1,6
BARRICADE,6,3,0,20,0,0,0
MATERIALS,3,2,0,1,21,1,1
SCAFFOLD,3,0,0,0,1,21,3
__FAILED__,0,0,0,0,0,0,0


'gpt-oss-120b'

,MACHINERY,DEMOLITION DUMPSTER,CRANE,BARRICADE,MATERIALS,SCAFFOLD,__FAILED__
MACHINERY,21,0,1,2,4,0,0
DEMOLITION DUMPSTER,0,29,0,0,0,0,0
CRANE,2,1,22,1,2,1,0
BARRICADE,5,6,1,15,2,0,0
MATERIALS,3,4,0,1,20,1,0
SCAFFOLD,0,1,0,0,2,25,0
__FAILED__,0,0,0,0,0,0,0


'muse-glimmer'

,MACHINERY,DEMOLITION DUMPSTER,CRANE,BARRICADE,MATERIALS,SCAFFOLD,__FAILED__
MACHINERY,25,0,0,1,1,1,0
DEMOLITION DUMPSTER,0,29,0,0,0,0,0
CRANE,2,0,22,0,4,1,0
BARRICADE,3,5,2,19,0,0,0
MATERIALS,1,4,0,2,21,1,0
SCAFFOLD,0,0,0,0,0,28,0
__FAILED__,0,0,0,0,0,0,0


,model,prompt_condition,n_examples,n_scored,accuracy,macro_f1,failed_predictions,prompt_tokens,completion_tokens
0,llama-4-scout,boundary_review_ten,10,172,0.726744,0.770819,19,281458.0,4325.0
1,gpt-oss-120b,boundary_review_ten,10,172,0.767442,0.763780,0,324970.0,15892.0
2,muse-glimmer,boundary_review_ten,10,172,0.837209,0.833534,0,318344.0,37882.0


,model,permit_id,description,actual,predicted,confidence,reason,error
3,llama-4-scout,DOMI-CS-2020-03505,"DP-2019-03308DEMOLITION OF 209 FIRST AVENUE, PITTSBURGH, PA 15222",BARRICADE,DEMOLITION DUMPSTER,1.00,Demolition indicated,None
5,llama-4-scout,DOMI-CS-2020-03667,"DEMOLISH AND REPLACE PLANTED MEDIAN ON FEDERAL STREET, INSTALLING GRASS BUFFER, AND CURB RAMPS ON FEDERAL STREET",BARRICADE,MACHINERY,0.90,operational work with unspecified equipment,None
11,llama-4-scout,DOMI-CS-2020-07320,A LIFT WILL BE USED TO REMOVE HOLIDAY WREATHS FROM THE EXTERIOR OF THE FIFTH AVE PLACE BUILDING.,MACHINERY,CRANE,1.00,LIFT WILL BE USED,None
18,llama-4-scout,DOMI-CS-2021-01396,REMOVAL OF OLD BUILDING MATERIALS INSIDE THE PROPERTY.,DEMOLITION DUMPSTER,__FAILED__,NaN,None,"Expecting ',' delimiter: line 1 column 51 (char 50)"
21,llama-4-scout,DOMI-CS-2021-01658,3 REVERSE WALL ANCHORS ON FRONT OF BUILDING,SCAFFOLD,MACHINERY,0.50,operational equipment for anchor installation,None
...,...,...,...,...,...,...,...,...
487,muse-glimmer,DOMI-CS-2025-12830,MASSARO CONSTRUCTION DOING REPAIRS ON THE INSIDE ELEVATOR (MEDICAL STRETCHER UP GRADE) AT THE BENEDUM THEATER 237 7TH ST. PGH PA ST. 15222 / DEMOLITION DUMPSTER WILL BE SET UP IN TRAVEL LANE (CURB LANE) BETWEEN LIBERTY AVE. TO PENN AVE. / DUMPSTER WILL BE MOVED WHEN THE BENEDUM HAS EVENTS / 1 LANE IN EACH DIRECTION WILL BE MAINTAINED / THIS IS AN ONGOING PROJECT / THE CONSTRUCTION IS STARTING AGAIN ON 10-18-25 / PRIOR PERMIT WAS DOMI-CS-2025-08064,BARRICADE,DEMOLITION DUMPSTER,0.85,Description explicitly states a demolition dumpster will be set up in the curb travel lane for the ongoing project.,None
490,muse-glimmer,DOMI-CS-2025-14154,BURNS & SCALO ROOFING WILL BE LIFTING ROOFING MATERIALS AT 5000 5TH AVE. APARTMENT BUILDING / PRIOR PERMIT WAS DOMI-CS-2025-12867,CRANE,MATERIALS,0.45,"Description states lifting roofing materials for roof work, indicating material handling/delivery rather than explicit crane, dumpster or scaffold.",None
501,muse-glimmer,DOMI-CS-2026-03778,"REMOVE AND REPLACE CONCRETE BACKWALLS ON HERRON AVE BRIDGE AT SEALSWAIVE FEES - CITY JOB, ERIN COWEN / TIM KINNETT",BARRICADE,DEMOLITION DUMPSTER,0.45,Remove and replace concrete backwalls implies demolition debris removal.,None
507,muse-glimmer,DOMI-CS-2026-07197,DRONE-POWERED WINDOW CLEANING ON THE RENAISSANCE HOTEL,BARRICADE,MACHINERY,0.70,"Drone-powered window cleaning is operational equipment use for facade work, similar to JLG window cleaning precedent.",None


In [5]:
baseline_path = PROJECT_ROOT / "category_results" / "classification_results.csv"
if baseline_path.exists():
    baseline = pd.read_csv(baseline_path)
    comparisons = []
    for (model, condition), group in baseline.groupby(["model", "prompt_condition"]):
        matched = group[group.permit_id.isin(test.permit_id)]
        if len(matched) != len(test) or matched.permit_id.nunique() != len(test):
            print("Skipping incomplete baseline:", model, condition)
            continue
        aligned = matched.set_index("permit_id").loc[test.permit_id]
        assert aligned.actual.tolist() == test[LABEL_COL].tolist()
        assert aligned.description.tolist() == test.work_description.tolist()
        scores = score_results(matched, CATEGORIES)
        current = summary[summary.model.eq(model)]
        comparisons.append({
            "model": model, "baseline_condition": condition, "n_scored": len(matched),
            "baseline_accuracy": scores["accuracy"], "baseline_macro_f1": scores["macro_f1"],
            "new_accuracy": current.accuracy.iloc[0] if len(current) else None,
            "accuracy_change": current.accuracy.iloc[0] - scores["accuracy"] if len(current) else None,
        })
    comparison = pd.DataFrame(comparisons)
    comparison.to_csv(RUN_DIR / "baseline_comparison.csv", index=False)
    display(comparison)
else:
    print("Original raw results CSV is unavailable locally; baseline experiments were not rerun.")


Original raw results CSV is unavailable locally; baseline experiments were not rerun.


In [6]:
# Compare the revised prompt with the previous completed OpenRouter run.
previous_paths = [p for p in OUTPUT_DIR.glob("*/classification_results.csv")
                  if p.parent != RUN_DIR]
if previous_paths:
    previous_path = max(previous_paths, key=lambda p: p.stat().st_mtime)
    previous_results = pd.read_csv(previous_path)
    rows = []
    for model in MODELS:
        before = previous_results[previous_results.model.eq(model)]
        after = results[results.model.eq(model)]
        if before.permit_id.duplicated().any() or set(before.permit_id) != set(test.permit_id):
            print("Skipping previous model with a different/incomplete evaluation set:", model)
            continue
        before = before.set_index("permit_id").loc[test.permit_id]
        assert before.actual.tolist() == test[LABEL_COL].tolist()
        assert before.description.tolist() == test.work_description.tolist()
        old_scores = score_results(before, CATEGORIES)
        new_scores = score_results(after, CATEGORIES)
        rows.append({
            "model": model, "previous_accuracy_percent": 100 * old_scores["accuracy"],
            "revised_accuracy_percent": 100 * new_scores["accuracy"],
            "change_percentage_points": 100 * (new_scores["accuracy"] - old_scores["accuracy"]),
            "previous_failures": old_scores["n_failures"], "revised_failures": new_scores["n_failures"],
        })
    revision_comparison = pd.DataFrame(rows)
    revision_comparison.to_csv(RUN_DIR / "prompt_revision_comparison.csv", index=False)
    display(revision_comparison)
else:
    print("No prior OpenRouter results available for comparison.")


,model,previous_accuracy_percent,revised_accuracy_percent,change_percentage_points,previous_failures,revised_failures
0,llama-4-scout,75.000000,72.674419,-2.325581,8,19
1,gpt-oss-120b,78.488372,76.744186,-1.744186,0,0
2,muse-glimmer,80.813953,83.720930,2.906977,0,0


In [7]:
# Accuracy for each model: failed predictions count as incorrect.
accuracy_by_model = (
    results.assign(correct=results["actual"].eq(results["predicted"]))
    .groupby("model", sort=False)
    .agg(correct_predictions=("correct", "sum"), total_predictions=("correct", "size"))
    .reset_index()
)
accuracy_by_model["accuracy_percent"] = (
    100 * accuracy_by_model["correct_predictions"] / accuracy_by_model["total_predictions"]
)
display(accuracy_by_model.style.format({"accuracy_percent": "{:.2f}%"}))
for row in accuracy_by_model.itertuples(index=False):
    print(f"{row.model}: {row.accuracy_percent:.2f}% accuracy "
          f"({row.correct_predictions}/{row.total_predictions} correct)")


,model,correct_predictions,total_predictions,accuracy_percent
0,llama-4-scout,125,172,72.67%
1,gpt-oss-120b,132,172,76.74%
2,muse-glimmer,144,172,83.72%


llama-4-scout: 72.67% accuracy (125/172 correct)
gpt-oss-120b: 76.74% accuracy (132/172 correct)
muse-glimmer: 83.72% accuracy (144/172 correct)
